# 🚀 Chapter 07-02b: Build an Agent That Searches, Decides & Acts

---

## 📖 The Mission

Build a LangGraph agent with a proper **agent loop** (cycle). The agent will:
1. Think about what to do
2. Call tools if needed
3. Loop back to think again
4. Stop when it has the answer

Plus: **checkpointing** for conversation memory!

---

## 🧪 Step 1: Setup & Tools

In [ ]:
# !pip install langgraph langchain-openai langchain-core python-dotenv

In [ ]:
import os
import json
from datetime import datetime
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
print("✅ All imports ready!")

In [ ]:
# Define tools
@tool
def search_web(query: str) -> str:
    """Search the web for current information. Use for facts, news, and data."""
    # Simulated search results
    results = {
        "python": "Python 3.13 released Oct 2024. Key features: improved error messages, free-threaded mode.",
        "langchain": "LangChain v0.3 is the latest stable release. Major change: Pydantic v2 migration.",
        "weather": "Current weather simulation: 72°F, partly cloudy.",
        "langgraph": "LangGraph is a library for building stateful agents with cycles and persistence.",
    }
    for key, val in results.items():
        if key in query.lower():
            return val
    return f"Search results for '{query}': No specific results found. Try a more specific query."

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Use for any calculations."""
    try:
        # Safe evaluation - only math
        allowed = set('0123456789+-*/().% ')
        if not all(c in allowed for c in expression):
            return "Error: Only basic math operations allowed."
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Calculation error: {str(e)}"

@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

tools = [search_web, calculate, get_current_time]
tool_map = {t.name: t for t in tools}
print(f"✅ {len(tools)} tools defined: {[t.name for t in tools]}")

## 🧪 Step 2: State & Nodes

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

# Node: Agent thinks and possibly calls tools
def agent_node(state: AgentState) -> dict:
    messages = [
        SystemMessage(content="You are a helpful assistant with access to search, calculator, and time tools. Use them when needed."),
        *state["messages"]
    ]
    response = llm.invoke(messages)
    return {"messages": [response]}

# Node: Execute tools
def tool_node(state: AgentState) -> dict:
    last_msg = state["messages"][-1]
    results = []
    for tc in last_msg.tool_calls:
        print(f"  🔧 Executing: {tc['name']}({tc['args']})")
        if tc["name"] in tool_map:
            try:
                result = tool_map[tc["name"]].invoke(tc["args"])
            except Exception as e:
                result = f"Error: {str(e)}"
        else:
            result = f"Unknown tool: {tc['name']}"
        results.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return {"messages": results}

# Router: Should we continue the loop?
def should_continue(state: AgentState) -> Literal["tools", "end"]:
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "end"

print("✅ Nodes and router defined!")

## 🧪 Step 3: Build Graph with Cycle

In [ ]:
# Build the graph
graph = StateGraph(AgentState)

graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

# THE CYCLE: agent → tools → agent (loop!)
graph.add_conditional_edges("agent", should_continue, {
    "tools": "tools",
    "end": END
})
graph.add_edge("tools", "agent")  # ← This creates the cycle!

# Compile with checkpointer for memory
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("✅ Graph compiled with cycle and checkpointer!")
try:
    print(app.get_graph().draw_ascii())
except Exception:
    print("Graph: START → agent ↔ tools → END")

## 🧪 Step 4: Test the Agent Loop

In [ ]:
# Helper to send messages
def ask(question: str, thread_id: str = "demo-thread"):
    config = {"configurable": {"thread_id": thread_id}}
    print(f"\n{'='*50}")
    print(f"🧑 User: {question}")
    print(f"{'='*50}")
    
    result = app.invoke(
        {"messages": [HumanMessage(content=question)]},
        config
    )
    final = result["messages"][-1].content
    print(f"\n🤖 Agent: {final[:500]}")
    return final

# Test 1: Simple question (no tools needed)
ask("What is 2 + 2?")

In [ ]:
# Test 2: Requires tool call (search)
ask("What's the latest version of Python?")

In [ ]:
# Test 3: Multi-tool (search + calculate)
ask("What time is it? And what's 15% of 250?")

In [ ]:
# Test 4: Conversation memory! (same thread_id)
ask("What did I just ask about?", thread_id="demo-thread")

## 🧪 Step 5: Inspect Checkpoints

In [ ]:
# View conversation history stored in checkpoint
config = {"configurable": {"thread_id": "demo-thread"}}
state = app.get_state(config)

print("📝 Stored messages in checkpoint:")
for msg in state.values["messages"]:
    role = type(msg).__name__.replace("Message", "")
    content = str(msg.content)[:80] if msg.content else "(tool calls)"
    print(f"  {role}: {content}")

## ✍️ Practice Challenge

Add `interrupt_before=["tools"]` to the graph compilation and implement a human-approval flow:
1. Agent decides to use a tool
2. Graph pauses
3. You review the planned tool call
4. Resume with `app.invoke(None, config)`

<details>
<summary>💡 Hint</summary>

```python
app_hitl = graph.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["tools"]
)
config = {"configurable": {"thread_id": "hitl-demo"}}

# First invoke — will pause before tools
result = app_hitl.invoke({"messages": [HumanMessage("Search for LangGraph")]}, config)
print("Paused! Review:", result["messages"][-1].tool_calls)

# Approve and resume
result = app_hitl.invoke(None, config)
print("Final:", result["messages"][-1].content)
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Cycle: tools → agent edge creates the agent loop.              │
├─────────────────────────────────────────────────────────────────────┤
│  📌 should_continue() decides: more tools or END.                  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 MemorySaver + thread_id = conversation persistence.            │
├─────────────────────────────────────────────────────────────────────┤
│  📌 get_state() inspects checkpoint contents.                      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 This is the standard LangGraph agent pattern — memorize it!    │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Advanced LangGraph — Multi-Agent Graphs & Streaming!

---

### 🎉 Your Agent Has a LOOP!
Think → Act → Observe → Repeat — the real agent pattern. 🚀